In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
from gmx_env import load_gromacs_env
load_gaussian_env()
print("_________________-")
load_gromacs_env()

In [ ]:
import os
import re
import pandas as pd

os.environ["PATH"] += os.pathsep + "/root/Gaussian16_Linux_AVX2/tar/g16"

def parse_itp_files(df_system):
    

    
    itp_atom_type_dict = {}

    
    header_pattern = re.compile(r'^;\s+Index\s+type\s+residue\s+resname\s+atom\s+cgnr\s+charge\s+mass')

    
    
    data_pattern = re.compile(
        r'^\s*(\d+)\s+'        
        r'(\S+)\s+'            
        r'(\d+)\s+'            
        r'(\S+)\s+'            
        r'(\S+)\s+'            
        r'(\d+)\s+'            
        r'([\-\+\d\.]+)\s+'    
        r'([\-\+\d\.]+)\s*$'   
    )

    
    for idx, row in df_system.iterrows():
        
        name = row['Name']

        
        itp_filename = f"{name}.itp"
        if not os.path.isfile(itp_filename):
            
            print("Public message removed for release.")
            continue

        
        itp_atom_type_list = []

        
        with open(itp_filename, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        
        header_index = None
        for i, line in enumerate(lines):
            if header_pattern.search(line):
                header_index = i
                break

        if header_index is None:
            
            print("Public message removed for release.")
            continue

        
        start_line_index = header_index + 1

        for line in lines[start_line_index:]:
            
            match_data = data_pattern.match(line)
            if match_data:
                atom_name = match_data.group(5)  
                atom_name = atom_name[:4]  
                itp_atom_type_list.append(atom_name)
            else:
                
                
                break

        
        itp_atom_type_dict[name] = itp_atom_type_list

    
    return itp_atom_type_dict


In [ ]:
df_system = pd.read_excel("System.xlsx")


itp_atom_type_dict = parse_itp_files(df_system)


for k, v in itp_atom_type_dict.items():
    print("Public message removed for release.")

In [ ]:
import os
import re
from collections import defaultdict

def parse_coordination_mol2_files(itp_atom_type_dict, coordination_structure_to_compute_path=None):
    

    
    if coordination_structure_to_compute_path is None:
        coordination_structure_to_compute_path = os.path.join(os.getcwd(), "coordination_structure_to_compute")

    
    if not os.path.isdir(coordination_structure_to_compute_path):
        print("Public message removed for release.")
        return {}

    
    coordination_mol_dict = {}

    
    mol2_files = [f for f in os.listdir(coordination_structure_to_compute_path) 
                  if f.endswith(".mol2")]

    
    if not mol2_files:
        print("Public message removed for release.")
        return {}

    
    
    
    reverse_map = {}
    for dict_key, dict_values in itp_atom_type_dict.items():
        for val in dict_values:
            reverse_map[val] = dict_key

    
    atom_block_pattern = re.compile(r'^@<TRIPOS>ATOM')

    
    for mol2_file in mol2_files:
        
        coordination_structure_name = os.path.splitext(mol2_file)[0]

        
        mol2_path = os.path.join(coordination_structure_to_compute_path, mol2_file)
        with open(mol2_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        
        coordination_structure_atom_type_list = []

        
        atom_block_start = None
        for i, line in enumerate(lines):
            if atom_block_pattern.match(line.strip()):
                atom_block_start = i
                break

        
        if atom_block_start is None:
            print("Public message removed for release.")
            continue

        
        i = atom_block_start + 1
        while i < len(lines):
            current_line = lines[i].strip()
            
            if not current_line or current_line.startswith('@<TRIPOS>'):
                break

            
            
            parts = current_line.split()
            if len(parts) < 2:
                
                break
            
            atom_type = parts[1]
            coordination_structure_atom_type_list.append(atom_type)

            i += 1

        
        

        
        
        composition_count = defaultdict(int)
        for atype in coordination_structure_atom_type_list:
            
            if atype in reverse_map:
                composition_key = reverse_map[atype]
                composition_count[composition_key] += 1
            else:
                
                
                print("Public message removed for release.")
                continue
        
        for key in composition_count:
            
            
            composition_count[key] = composition_count[key]
        
        
        coordination_mol_number_list = dict(composition_count)
            
        
        coordination_mol_dict[coordination_structure_name] = coordination_mol_number_list

    
    return coordination_mol_dict


In [ ]:

coordination_mol_dict = parse_coordination_mol2_files(itp_atom_type_dict)


for structure_name, composition in coordination_mol_dict.items():
    print("Public message removed for release.")

In [ ]:
coordination_mol_dict

In [ ]:

import os
import pandas as pd

def compute_coordination_system_charge(coordination_mol_dict, df_system_path='System.xlsx'):
    

    
    if not os.path.isfile(df_system_path):
        print("Public message removed for release.")
        return {}

    
    df_system = pd.read_excel(df_system_path)

    
    coordination_system_charge_dict = {}

    
    for coordination_name, sub_dict in coordination_mol_dict.items():
        
        system_charge = 0

        
        for mol_name, mol_number in sub_dict.items():
            
            
            net_charge_values = df_system.loc[df_system['Name'] == mol_name, 'Net Charge'].values
            if len(net_charge_values) > 0:
                net_charge = net_charge_values[0]
            else:
                net_charge = 0
                print("Public message removed for release.")

            
            system_charge += mol_number * net_charge

        
        coordination_system_charge_dict[coordination_name] = system_charge

    return coordination_system_charge_dict


In [ ]:

coordination_system_charge_dict = compute_coordination_system_charge(coordination_mol_dict, 'System.xlsx')
print(coordination_system_charge_dict)

In [ ]:

import os
import re

def parse_top_file(top_file_path="system.top"):
    

    if not os.path.isfile(top_file_path):
        print("Public message removed for release.")
        return {}

    
    periodic_table = {
        1: 'H', 2: 'He', 3: 'Li', 4: 'Be', 5: 'B', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 10: 'Ne',
        11: 'Na', 12: 'Mg', 13: 'Al', 14: 'Si', 15: 'P', 16: 'S', 17: 'Cl', 18: 'Ar', 19: 'K', 20: 'Ca',
        21: 'Sc', 22: 'Ti', 23: 'V', 24: 'Cr', 25: 'Mn', 26: 'Fe', 27: 'Co', 28: 'Ni', 29: 'Cu', 30: 'Zn',
        31: 'Ga', 32: 'Ge', 33: 'As', 34: 'Se', 35: 'Br', 36: 'Kr', 37: 'Rb', 38: 'Sr', 39: 'Y', 40: 'Zr',
        41: 'Nb', 42: 'Mo', 43: 'Tc', 44: 'Ru', 45: 'Rh', 46: 'Pd', 47: 'Ag', 48: 'Cd', 49: 'In', 50: 'Sn',
        51: 'Sb', 52: 'Te', 53: 'I', 54: 'Xe', 55: 'Cs', 56: 'Ba', 57: 'La', 58: 'Ce', 59: 'Pr', 60: 'Nd',
        61: 'Pm', 62: 'Sm', 63: 'Eu', 64: 'Gd', 65: 'Tb', 66: 'Dy', 67: 'Ho', 68: 'Er', 69: 'Tm', 70: 'Yb',
        71: 'Lu', 72: 'Hf', 73: 'Ta', 74: 'W', 75: 'Re', 76: 'Os', 77: 'Ir', 78: 'Pt', 79: 'Au', 80: 'Hg',
        81: 'Tl', 82: 'Pb', 83: 'Bi', 84: 'Po', 85: 'At', 86: 'Rn', 87: 'Fr', 88: 'Ra', 89: 'Ac', 90: 'Th',
        91: 'Pa', 92: 'U', 93: 'Np', 94: 'Pu', 95: 'Am', 96: 'Cm', 97: 'Bk', 98: 'Cf', 99: 'Es', 100: 'Fm',
        101: 'Md', 102: 'No', 103: 'Lr', 104: 'Rf', 105: 'Db', 106: 'Sg', 107: 'Bh', 108: 'Hs', 109: 'Mt',
        110: 'Ds', 111: 'Rg', 112: 'Cn', 113: 'Nh', 114: 'Fl', 115: 'Mc', 116: 'Lv', 117: 'Ts', 118: 'Og'
    }

    
    header_pattern = re.compile(
        r'^;\s+name\s+at\.num\s+mass\s+charge\s+ptype\s+sigma\s*\(nm\)\s+epsilon\s*\(kJ/mol\)'
    )

    
    
    
    data_pattern = re.compile(
        r'^\s*(\S+)\s+(\d+)\s+([\-\+\d\.E]+)\s+([\-\+\d\.E]+)\s+(\S+)\s+([\-\+\d\.E]+)\s+([\-\+\d\.E]+)'
    )

    
    with open(top_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    
    header_index = None
    for i, line in enumerate(lines):
        if header_pattern.match(line.strip()):
            header_index = i
            break

    
    if header_index is None:
        print("Public message removed for release.")
        return {}

    
    start_line_index = header_index + 1

    
    atom_name_to_number = {}

    i = start_line_index
    while i < len(lines):
        line_strip = lines[i].strip()
        
        if not line_strip:
            break

        
        match_data = data_pattern.match(line_strip)
        if match_data:
            atom_name = match_data.group(1)       
            atomic_number_str = match_data.group(2)  
            atomic_number = int(atomic_number_str)

            
            if atom_name not in atom_name_to_number:
                atom_name_to_number[atom_name] = atomic_number
        else:
            
            break

        i += 1

    
    
    atom_name_to_element = {}
    for name, at_num in atom_name_to_number.items():
        if at_num in periodic_table:
            atom_name_to_element[name] = periodic_table[at_num]
        else:
            
            
            atom_name_to_element[name] = f"Unknown({at_num})"

    return atom_name_to_element

In [ ]:
atom_name_to_element = parse_top_file("system.top")
print("Public message removed for release.")
for atom_name, element_symbol in atom_name_to_element.items():
    print(f"{atom_name:10s} => {element_symbol}")

In [ ]:

import os
import re

def create_gaussian_input_files(
    coordination_mol_dict,
    atom_name_to_element,
    coordination_system_charge_dict,
    coordination_structure_to_compute_path=None,
    nproc=10,
    mem=20
):
    

    
    if coordination_structure_to_compute_path is None:
        coordination_structure_to_compute_path = os.path.join(
            os.getcwd(), "coordination_structure_to_compute"
        )
    
    
    os.makedirs(coordination_structure_to_compute_path, exist_ok=True)

    
    atom_block_pattern = re.compile(r'^@<TRIPOS>ATOM')

    
    
    
    for coordination_name, charge in coordination_system_charge_dict.items():
        
        mol2_filename = f"{coordination_name}.mol2"
        mol2_path = os.path.join(coordination_structure_to_compute_path, mol2_filename)
        
        
        if not os.path.isfile(mol2_path):
            print("Public message removed for release.")
            continue
        
        
        with open(mol2_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        
        atom_block_start = None
        for i, line in enumerate(lines):
            if atom_block_pattern.match(line.strip()):
                atom_block_start = i
                break
        
        if atom_block_start is None:
            print("Public message removed for release.")
            continue
        
        
        coordinates = ""
        i = atom_block_start + 1
        while i < len(lines):
            current_line = lines[i].rstrip("\n")  
            if not current_line.strip() or current_line.startswith('@<TRIPOS>'):
                
                break
            
            
            
            
            parts = current_line.split()
            if len(parts) < 5:
                
                break
            
            
            forcefield_atom_name = parts[1]  
            x_coord = parts[2]              
            y_coord = parts[3]              
            z_coord = parts[4]              

            
            pattern = r'^(.*)_(\d+)$'
            match = re.match(pattern, forcefield_atom_name)
            if match:
                
                forcefield_atom_name = match.group(1)
            
            
            
            element = atom_name_to_element.get(forcefield_atom_name, forcefield_atom_name)
            
            
            coordinates += f"{element:2s}     {x_coord}   {y_coord}   {z_coord}\n"

            i += 1
        
        
        gjf_filename = f"{coordination_name}.gjf"
        gjf_path = os.path.join(coordination_structure_to_compute_path, gjf_filename)
        
        
        chk_filename = f"{coordination_name}.chk"
        chk_path = os.path.join(coordination_structure_to_compute_path, chk_filename)
        chk_abs_path = os.path.abspath(chk_path)
        
        
        with open(gjf_path, 'w', encoding='utf-8') as gf:
            gf.write(f"%nprocshared={nproc}\n")
            gf.write(f"%mem={mem}GB\n")
            gf.write(f"%chk={chk_abs_path}\n")
            gf.write("# M062X/6-311+G(2d,p) em=gd3 \n\n")  
            gf.write(f"{coordination_name}\n\n")
            gf.write(f"{charge} 1\n")  
            gf.write(coordinates)
            gf.write("\n\n")
        
        print("Public message removed for release.")

In [ ]:

create_gaussian_input_files(
    coordination_mol_dict,
    atom_name_to_element,
    coordination_system_charge_dict,
    coordination_structure_to_compute_path=None,
    nproc=10,
    mem=20)

In [ ]:

import pandas as pd

def build_coordination_df(df_system, coordination_mol_dict):
    

    
    name_list = df_system['Name'].unique().tolist()

    
    columns = ['coordination_name'] + name_list

    
    data_rows = []

    
    for coordination_name, sub_dict in coordination_mol_dict.items():
        
        current_row = [coordination_name]

        
        for nm in name_list:
            
            val = sub_dict.get(nm, 0)
            current_row.append(val)

        
        data_rows.append(current_row)

    
    coordination_df = pd.DataFrame(data=data_rows, columns=columns)

    
    return coordination_df

In [ ]:

coordination_df = build_coordination_df(df_system, coordination_mol_dict)
print(coordination_df)

In [ ]:

import os
import shutil
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
import pandas as pd



def run_one_gaussian(name, coordination_structure_to_compute_path=None):
    
    
    if coordination_structure_to_compute_path is None:
        coordination_structure_to_compute_path = os.path.join(
            os.getcwd(), "coordination_structure_to_compute"
        )
    
    
    gjf_file = os.path.abspath(
        os.path.join(coordination_structure_to_compute_path, f"{name}.gjf")
    )
    out_file = os.path.abspath(
        os.path.join(coordination_structure_to_compute_path, f"{name}.out")
    )
    chk_file = os.path.abspath(
        os.path.join(coordination_structure_to_compute_path, f"{name}.chk")
    )
    
    
    success_dir = os.path.abspath(os.path.join(coordination_structure_to_compute_path, 'success'))
    failure_dir = os.path.abspath(os.path.join(coordination_structure_to_compute_path, 'failure'))
    os.makedirs(success_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)

    if not os.path.exists(gjf_file):
        print("Public message removed for release.")
        
        return (out_file, False)

    try:

        
        subprocess.run(['g16', gjf_file, out_file], check=True)

        
        if os.path.exists(out_file):
            with open(out_file, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                
                if 'Normal termination' in content:
                    
                    shutil.move(gjf_file, os.path.join(success_dir, os.path.basename(gjf_file)))
                    shutil.move(out_file, os.path.join(success_dir, os.path.basename(out_file)))
                    if os.path.exists(chk_file):
                        shutil.move(chk_file, os.path.join(success_dir, os.path.basename(chk_file)))
                    return (out_file, True)
                else:
                    
                    shutil.move(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
                    shutil.move(out_file, os.path.join(failure_dir, os.path.basename(out_file)))
                    if os.path.exists(chk_file):
                        shutil.move(chk_file, os.path.join(failure_dir, os.path.basename(chk_file)))
                    return (out_file, False)
        else:
            
            if os.path.exists(gjf_file):
                shutil.move(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
            if os.path.exists(chk_file):
                shutil.move(chk_file, os.path.join(failure_dir, os.path.basename(chk_file)))
            return (out_file, False)

    except subprocess.CalledProcessError:
        
        if os.path.exists(gjf_file):
            shutil.move(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
        if os.path.exists(out_file):
            shutil.move(out_file, os.path.join(failure_dir, os.path.basename(out_file)))
        if os.path.exists(chk_file):
            shutil.move(chk_file, os.path.join(failure_dir, os.path.basename(chk_file)))
        return (out_file, False)

def run_gaussian_calculations(coordination_df, 
                              coordination_structure_to_compute_path=None, 
                              max_tasks=2):
    

    if coordination_structure_to_compute_path is None:
        coordination_structure_to_compute_path = os.path.join(
            os.getcwd(), "coordination_structure_to_compute"
        )
    
    
    success_dir = os.path.join(coordination_structure_to_compute_path, 'success')
    failure_dir = os.path.join(coordination_structure_to_compute_path, 'failure')
    os.makedirs(success_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)

    
    names_to_run = coordination_df['coordination_name'].unique().tolist()

    
    futures = []
    with ProcessPoolExecutor(max_workers=max_tasks) as executor:
        for name in names_to_run:
            futures.append(executor.submit(run_one_gaussian, name))

        
        for future in as_completed(futures):
            out_file, success_flag = future.result()
            if success_flag:
                print(f"[SUCCESS] {out_file}")
            else:
                print(f"[FAILURE] {out_file}")

In [ ]:
run_gaussian_calculations(coordination_df, 
                          coordination_structure_to_compute_path=None, 
                          max_tasks=2)

In [ ]:

def extract_energy_from_out(out_file):
    energy = None
    file_path = out_file
    with open(file_path, 'r') as file:
        lines = file.readlines()
        energy_line = ''
        for i, line in enumerate(lines):
            
            line = line.strip().replace('\n', '')
            
            if line.strip().endswith('H') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('F='):
                    
                    line = line.strip() + next_line.strip()
            
            if line.strip().endswith('HF') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('='):
                    
                    line = line.strip() + next_line.strip()
            
            if 'HF=' in line:
                
                start = line.find('HF=') + 3  
                energy_line = line[start:]
                
                if '\\' in energy_line:
                    energy = energy_line.split('\\')[0].strip()
                    return energy
                else:
                    
                    for j in range(i+1, len(lines)):
                        energy_line += lines[j].strip()
                        if '\\' in lines[j]:
                            energy = energy_line.split('\\')[0].strip()
                            return energy
    return energy  



def extract_properties_from_energycal(coordination_df, success_dir=None):
    
    df = coordination_df

    if success_dir is None:
        success_dir = os.path.join(os.getcwd(), "coordination_structure_to_compute", "success")
    
    if 'coordination_energy(Hatree)' not in df.columns:
        
        df['coordination_energy(Hatree)'] = 0.0
    
    
    for filename in os.listdir(success_dir):
        if filename.endswith('.out'):
            
            coordination_name = filename.rsplit('.', 1)[0]
            
            energy = extract_energy_from_out(os.path.join(success_dir, filename))
            
            
            energy = float(energy) if energy else 0.0
            
            
            
            df.loc[df['coordination_name'] == coordination_name, 'coordination_energy(Hatree)'] = float(energy)

    return df

In [ ]:
df = extract_properties_from_energycal(coordination_df, success_dir=None)

In [ ]:
df

In [ ]:

import pandas as pd

def compute_solvent_cage_escape_energy(df_system, coordination_df):
    
    
    
    name_energy_dict = dict(zip(df_system["Name"], df_system["Energy(Hatree)"]))

    
    col_name_list = coordination_df.columns.tolist()
    
    
    if "coordination_energy(Hatree)" not in col_name_list:
        raise ValueError("Public message removed for release.")
    
    
    total_ligand_col = "total_ligand_energy(Hatree)"
    escape_energy_col = "solvent_cage_escape_energy(Hatree)"
    escape_energy_kj_col = "solvent_cage_escape_energy(kJ/mol)"
    
    
    for col in [total_ligand_col, escape_energy_col, escape_energy_kj_col]:
        if col in coordination_df.columns:
            coordination_df.drop(columns=col, inplace=True)
    
    
    coordination_df[total_ligand_col] = 0.0
    coordination_df[escape_energy_col] = 0.0
    coordination_df[escape_energy_kj_col] = 0.0
    
    
    hartree_to_kjmol = 2625.5
    
    for idx, row in coordination_df.iterrows():
        total_ligand_energy = 0.0
        
        
        for col_name in col_name_list:
            if col_name in name_energy_dict:
                
                mol_count = row[col_name]
                if pd.notna(mol_count) and mol_count != 0:
                    total_ligand_energy += mol_count * name_energy_dict[col_name]
        
        coordination_df.at[idx, total_ligand_col] = total_ligand_energy
        
        
        coord_energy = row["coordination_energy(Hatree)"]
        
        
        solvent_cage_escape = coord_energy - total_ligand_energy
        coordination_df.at[idx, escape_energy_col] = solvent_cage_escape
        
        
        coordination_df.at[idx, escape_energy_kj_col] = solvent_cage_escape * hartree_to_kjmol
    
    return coordination_df

In [ ]:

df_system = pd.read_excel("System.xlsx")
coordination_df = compute_solvent_cage_escape_energy(df_system, coordination_df)
print(coordination_df)

In [ ]:

import pandas as pd


def calculate_negative_median(coordination_df):
    
    
    negative_values = coordination_df[
        coordination_df['solvent_cage_escape_energy(kJ/mol)'] <= 0
    ]['solvent_cage_escape_energy(kJ/mol)']

    
    if negative_values.empty:
        return None

    
    sorted_values = negative_values.sort_values()

    
    solvent_cage_escape_energy = sorted_values.median()

    return -solvent_cage_escape_energy 


In [ ]:

escape_energy = calculate_negative_median(coordination_df)
print("Public message removed for release.")

In [ ]:


df_system['solvent_escape_energy(kJ/mol)'] = escape_energy
df_system.to_excel("System.xlsx",index=None)
df_system

In [ ]:
import zipfile
import shutil


def collect_and_compress_files(source_dir):
    
    target_extensions = ('.tif', '.gif', '.mp4', '.mol2')

    
    all_results_dir = os.path.join(source_dir, 'all_results')
    if not os.path.exists(all_results_dir):
        os.makedirs(all_results_dir)
    
    
    exact_files = ["prod_NVT.xtc", "prod_NVT.gro"]

    
    for file in os.listdir(source_dir):
        source_file = os.path.join(source_dir, file)
        
        
        pattern = (
            r'^.*_(?:rdf_cn|rdf|msd)\.xvg$'    
            r'|^hbnum_.*\.xvg$'               
            r'|^hblife_.*\.xvg$'              
        )
        
        if os.path.isfile(source_file) and file in exact_files:
            destination_file = os.path.join(all_results_dir, file)
            if os.path.abspath(source_file) != os.path.abspath(destination_file):
                shutil.copy2(source_file, destination_file)
            continue  

        if os.path.isfile(source_file) and (file.endswith(target_extensions) or re.match(pattern, file)):
            destination_file = os.path.join(all_results_dir, file)
            
            if os.path.abspath(source_file) != os.path.abspath(destination_file):
                shutil.copy2(source_file, destination_file)
        
        
        elif os.path.isdir(source_file) and file == 'Gaussian':
            target_gaussian_folder = os.path.join(all_results_dir, 'Gaussian')
            
            if not os.path.exists(target_gaussian_folder):
                shutil.copytree(source_file, target_gaussian_folder)
    
    
    zip_filename = os.path.join(source_dir, 'all_results.zip')
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(all_results_dir):
            for file in files:
                
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, all_results_dir))
    
    print("Public message removed for release.")


source_dir = '.'  
collect_and_compress_files(source_dir)